In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy samplomatic
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Ejemplos de Executor

<Accordion>
<AccordionItem title="Versiones de paquetes">

El código de esta página fue desarrollado con los siguientes requisitos.
Recomendamos usar estas versiones o versiones más recientes.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
samplomatic~=0.18.0
```
</AccordionItem>
</Accordion>

Los ejemplos en esta sección ilustran algunas formas comunes de usar la primitiva Executor. Antes de ejecutar estos ejemplos, sigue las instrucciones en [Instalar Qiskit](/guides/install-qiskit) y [Inicio rápido con Executor](/guides/directed-execution-model).

## Antes de comenzar
Algunos de los ejemplos de código en esta página usan `samplex`, que es parte del paquete Samplomatic. Por lo tanto, antes de ejecutar esos bloques de código, debes instalar Samplomatic, como se muestra en el siguiente bloque de código. Para más información, consulta la [documentación de Samplomatic](https://qiskit.github.io/samplomatic).

In [1]:
from qiskit.circuit import Parameter, QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, Executor
from qiskit_ibm_runtime.quantum_program import QuantumProgram
from qiskit.transpiler import generate_preset_pass_manager
import numpy as np
from samplomatic import build
from samplomatic.transpiler import generate_boxing_pass_manager

# Generate the circuit
circuit = QuantumCircuit(3)
circuit.h(0)
circuit.h(1)
circuit.cz(0, 1)
circuit.h(1)
circuit.h(2)
circuit.cz(1, 2)
circuit.h(2)
circuit.rz(Parameter("theta"), 0)
circuit.rz(Parameter("phi"), 1)
circuit.rz(Parameter("lam"), 2)
circuit.measure_all()

## Ejemplo: Circuito parametrizado
Este ejemplo ilustra cómo agregar elementos de circuito con parámetros, así como cómo agregar elementos samplex. Consiste en estos pasos:
1. Configurar el circuito: Generar y transpilar el circuito objetivo.
2. Preparar un samplex: Agrupar los gates y las mediciones en cajas anotadas y generar el par de circuito template y samplex.
3. Ejecutar: Agregar un elemento de circuito y un elemento samplex a un `QuantumProgram` y ejecutar ambos en un único job.

### Configurar el circuito
Preparar un estado GHZ de tres qubits, rotar los qubits alrededor del eje de Pauli-Z, y medir los qubits en la base computacional.

In [2]:
# Initialize the service and choose a backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Transpile the circuit to ISA
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=3
)
isa_circuit = preset_pass_manager.run(circuit)

Especificar el backend y transpilar el circuito para usar únicamente las instrucciones admitidas por el QPU (denominado circuito de arquitectura de conjunto de instrucciones (ISA)).

In [3]:
boxing_pm = generate_boxing_pass_manager(
    # Add gate twirling
    enable_gates=True,
    # Add measurement twirling
    enable_measures=True,
)

boxed_circuit = boxing_pm.run(isa_circuit)

### Preparar el samplex
Usar la función de conveniencia `generate_boxing_pass_manager` y sus parámetros de twirling para agrupar los gates de dos qubits y las mediciones en cajas y aplicar anotaciones de twirling.

In [4]:
# Build the template circuit and the samplex
template_circuit, samplex = build(boxed_circuit)

Usar el método `build` para generar el circuito template y el samplex.

In [5]:
# Generate a quantum program
program = QuantumProgram(shots=1024)

### Ejecutar los circuitos
Executor ejecuta objetos `QuantumProgram`. Cada `QuantumProgram` puede contener varios elementos. Este ejemplo agrega un elemento de circuito y un elemento samplex para la ejecución. Para obtener todos los detalles, consulta [Entrada y salida del Executor](/guides/executor-input-output).

El primer paso es inicializar un programa vacío, solicitando `1024` shots para cada configuración de cada elemento.

In [6]:
# Append the circuit and the parameter values to the program
program.append_circuit_item(
    isa_circuit,
    circuit_arguments=np.random.rand(10, 3),  # 10 sets of parameter values
)

Agregar el elemento de circuito al `QuantumProgram`. Este elemento de circuito consta de dos partes: el circuito ISA y 10 conjuntos de sus valores de parámetros.

In [7]:
# Append the template circuit and samplex as a samplex item
program.append_samplex_item(
    template_circuit,
    samplex=samplex,
    samplex_arguments={
        "parameter_values": np.random.rand(
            10, 3
        ),  # 10 sets of parameter values
    },
    shape=(2, 14, 10),
)

Agregar el elemento samplex al `QuantumProgram` con estos argumentos:
- El circuito template y el samplex generados por la función `build`
- Diez conjuntos de valores de parámetros para el circuito original
- El número de randomizaciones a realizar

In [8]:
# initialize an Executor with default options
executor = Executor(mode=backend)

# Submit the job
job = executor.run(program)

# Retrieve the result
result = job.result()

### Ejecutar el job de Executor

In [9]:
# Access the results of the classical register of task #0, the CircuitItem
result_0 = result[0]["meas"]

# Access the results of the classical register of task #1, the SamplexItem
result_1 = result[1]["meas"]

Recuperar el resultado para cada tarea.

In [10]:
from qiskit_ibm_runtime import QiskitRuntimeService, Executor
from qiskit_ibm_runtime.quantum_program import QuantumProgram
from qiskit.circuit import QuantumCircuit, Parameter
from qiskit.transpiler import generate_preset_pass_manager
from samplomatic.transpiler import generate_boxing_pass_manager
from samplomatic import build

# Initialize the service and choose a backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Prepare a circuit

num_qubits = 10
num_layers = 10

qubits = list(range(num_qubits))
circuit = QuantumCircuit(num_qubits)

for layer_idx in range(num_layers):
    circuit.rx(Parameter(f"theta_{layer_idx}"), qubits)
    for i in range(num_qubits // 2):
        circuit.cz(qubits[2 * i], qubits[2 * i + 1])

    circuit.rx(Parameter(f"phi_{layer_idx}"), qubits)
    for i in range(num_qubits // 2 - 1):
        circuit.cz(qubits[2 * i] + 1, qubits[2 * i + 1] + 1)

circuit.draw("mpl", scale=0.35, fold=100)

<Image src="../docs/images/guides/executor-examples/extracted-outputs/f9e93b2c-154a-4d09-872d-f770bcc669c4-0.svg" alt="Output of the previous code cell" />

## Ejemplo: Realizar PEC
Este ejemplo muestra cómo usar un elemento samplex para realizar la cancelación de errores probabilista ([PEC](/guides/error-mitigation-and-suppression-techniques#pec)) para la mitigación de errores.

Considera una versión espejo de un circuito con diez qubits y dos capas únicas de gates CX. Estas son las tareas principales:
- Ejecutar el circuito con twirling.
- Ejecutar el circuito con mitigación PEC, como en el artículo ["Probabilistic error cancellation with sparse Pauli-Lindblad models on noisy quantum processors"](https://arxiv.org/abs/2201.09866).

El pipeline consiste en estos pasos:
1. Configuración: Generar el circuito objetivo y agrupar sus operaciones en cajas.
2. Aprendizaje: Aprender el ruido de las instrucciones que queremos mitigar con PEC.
3. Ejecución: Ejecutar el circuito en un backend.
4. Análisis: Post-procesar y analizar los resultados.

A modo de comparación, ejecutaremos este circuito espejo dos veces. Una vez con solo Pauli-twirling aplicado, y otra vez con la mitigación PEC aplicada.

> **Note:** El uso para este ejemplo es de aproximadamente 10 minutos en un procesador Heron r2.

### Configurar el circuito
Elegir un backend y preparar un circuito de 10 qubits.

In [11]:
mirror_circuit = circuit.compose(circuit.inverse())
mirror_circuit.measure_all()

mirror_circuit.draw("mpl", scale=0.35, fold=100)

<Image src="../docs/images/guides/executor-examples/extracted-outputs/f8ac3f75-88ca-40f9-8382-6a427303bb8e-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/executor-examples/extracted-outputs/f9e93b2c-154a-4d09-872d-f770bcc669c4-0.svg)

Combinar el circuito con su inverso para crear un circuito espejo.

In [12]:
import numpy as np

parameter_values = np.random.rand(mirror_circuit.num_parameters)

![Output of the previous code cell](../docs/images/guides/executor-examples/extracted-outputs/f8ac3f75-88ca-40f9-8382-6a427303bb8e-0.svg)

Establecer algunos valores de parámetros:

In [13]:
preset_pass_manager = generate_preset_pass_manager(
    backend=backend,
    optimization_level=3,
)

isa_circuit = preset_pass_manager.run(mirror_circuit)

Usar el pass manager para transpilar el circuito a un circuito ISA.

In [14]:
# Pass manager used to create twirled-annotated boxes.
boxing_pm = generate_boxing_pass_manager(
    enable_gates=True,
    enable_measures=True,
)

mirror_circuit_twirl = boxing_pm.run(isa_circuit)

# Pass manager used to create a new boxed circuit with
# both Twirl and InjectNoise annotations.
boxing_pm = generate_boxing_pass_manager(
    enable_gates=True,
    enable_measures=True,
    inject_noise_targets="gates",  # no measurement mitigation
    inject_noise_strategy="uniform_modification",
)

mirror_circuit_pec = boxing_pm.run(isa_circuit)

A continuación, agrupar los gates y las mediciones en cajas anotadas. Puedes hacerlo manualmente o usar la función `generate_boxing_pass_manager` de Samplomatic para mayor comodidad. El primer circuito solo tendrá twirling aplicado y por lo tanto solo necesita la anotación `Twirl`. El segundo circuito se ejecutará con mitigación PEC completa y necesita las anotaciones `Twirl` e `InjectNoise`.

In [15]:
from samplomatic.utils import find_unique_box_instructions

unique_box_instructions = find_unique_box_instructions(
    mirror_circuit_pec.data
)
assert len(unique_box_instructions) == 3

### Aprender el ruido
Para minimizar el número de experimentos de aprendizaje del ruido, identificar las instrucciones únicas en el segundo circuito (el que tiene cajas anotadas con `InjectNoise`). Para definir la unicidad, dos instrucciones de caja son iguales si ambas de las siguientes son verdaderas:
- Su contenido es igual, salvo por los gates de un solo qubit.
- Su anotación `Twirl` es igual (cualquier otra anotación se ignora).

Esto lleva a tres instrucciones únicas, a saber las cajas de gates pares e impares, y la caja de medición final.

In [16]:
from qiskit_ibm_runtime.noise_learner_v3 import NoiseLearnerV3

learner = NoiseLearnerV3(backend)

learner.options.shots_per_randomization = 128
learner.options.num_randomizations = 32
learner.options.layer_pair_depths = [0, 1, 2, 4, 16, 32]

learner_job = learner.run(unique_box_instructions)

learner_job.job_id()
learner_result = learner_job.result()

Inicializar un `NoiseLearnerV3`, elegir los parámetros de aprendizaje configurando sus opciones, y ejecutar un job de aprendizaje del ruido.

In [17]:
noise_maps = learner_result.to_dict(
    instructions=unique_box_instructions, require_refs=False
)

Convertir `result` al objeto requerido por el samplex usando el método `result.to_dict`.

In [18]:
from qiskit_ibm_runtime.quantum_program import QuantumProgram

# Initialize an empty QuantumProgram
program = QuantumProgram(shots=1000)

### Ejecutar los circuitos
`Executor` ejecuta objetos `QuantumProgram`. Cada `QuantumProgram` puede contener varios *elementos*, que se añaden al programa. Cada elemento es una tarea que el programa debe realizar.

Inicializar un programa vacío, solicitando `1000` shots para cada configuración de cada elemento.

In [19]:
template_twirl, samplex_twirl = build(mirror_circuit_twirl)

program.append_samplex_item(
    template_twirl,
    samplex=samplex_twirl,
    samplex_arguments={"parameter_values": parameter_values},
    shape=(900,),
)

A continuación, construir el circuito template y el samplex para `mirror_circuit_twirl` y añadirlos al programa. También solicitar `900` randomizaciones al samplex. Esto significa que el samplex generará `900` conjuntos de parámetros, y cada conjunto se ejecutará `1000` veces (el número de shots) en el QPU.

Esta es la primera tarea del programa (resultado 0).

In [20]:
template_pec, samplex_pec = build(mirror_circuit_pec)

program.append_samplex_item(
    template_pec,
    samplex=samplex_pec,
    samplex_arguments={
        "parameter_values": parameter_values,
        "pauli_lindblad_maps": noise_maps,
        "noise_scales": {
            ref: -1.0 for ref in noise_maps
        },  # Set the scales to -1 for PEC
    },
    shape=(900,),
)

Del mismo modo, añadir el circuito template y el samplex construidos para `mirror_circuit_pec`, solicitando `900` randomizaciones. Esta es la segunda tarea del programa (resultado 1).

In [21]:
from qiskit_ibm_runtime.executor import Executor

executor = Executor(backend)
executor_job = executor.run(program)

executor_job.job_id()

executor_results = executor_job.result()
executor_results

twirl_result = executor_results[0]

print(f"Twirl result keys:\n {list(twirl_result.keys())}\n")
print(f"Shape of results: {twirl_result['meas'].shape}")

pec_result = executor_results[1]

print(f"PEC result keys:\n {list(pec_result.keys())}\n")
print(f"Shape of results: {pec_result['meas'].shape}")

Twirl result keys:
 ['meas', 'measurement_flips.meas']

Shape of results: (900, 1000, 10)
PEC result keys:
 ['meas', 'measurement_flips.meas', 'pauli_signs']

Shape of results: (900, 1000, 10)


Importar `Executor` y enviar un job.

In [22]:
# Undo measurement twirling
twirl_result_unflipped = (
    twirl_result["meas"] ^ twirl_result["measurement_flips.meas"]
)

# Calculate the expectation values of single-qubit Z operators
exp_vals = 1 - 2 * twirl_result_unflipped.mean(axis=1).mean(axis=0)

for qubit, val in enumerate(exp_vals):
    print(f"Qubit {qubit} -> {np.round(val, 2)}")

Qubit 0 -> 0.77
Qubit 1 -> 0.76
Qubit 2 -> 0.66
Qubit 3 -> 0.71
Qubit 4 -> 0.69
Qubit 5 -> 0.67
Qubit 6 -> 0.62
Qubit 7 -> 0.59
Qubit 8 -> 0.62
Qubit 9 -> 0.68


In [23]:
# Undo measurement twirling
pec_result_unflipped = (
    pec_result["meas"] ^ pec_result["measurement_flips.meas"]
)

# Calculate the signs for PEC mitigation
signs = np.prod((-1) ** pec_result["pauli_signs"], axis=-1)
signs = signs.reshape((signs.shape[0], 1))

# Calculate the expectation values of single-qubit Z operators as required by
# PEC mitigation
exp_vals = 1 - (2 * pec_result_unflipped.mean(axis=1) * signs).mean(axis=0)

for qubit, val in enumerate(exp_vals):
    print(f"Qubit {qubit} -> {np.round(val, 2)}")

Qubit 0 -> 0.98
Qubit 1 -> 0.99
Qubit 2 -> 0.96
Qubit 3 -> 0.98
Qubit 4 -> 0.98
Qubit 5 -> 0.98
Qubit 6 -> 0.98
Qubit 7 -> 0.95
Qubit 8 -> 0.95
Qubit 9 -> 0.94


### Analizar los resultados
Por último, post-procesar los resultados para estimar los valores esperados de los operadores de Pauli-Z de un solo qubit actuando sobre cada uno de los diez qubits activos (valor esperado: `1.0`).